# Libraries

In [ ]:
import os
import csv
import time
import yaml
import shutil
import random
import kagglehub
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt 
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# Utils

### DOWNLOAD=True if you want to import into Input in Kaggle Notebook and download in Google Colab

CHANGE `data_root`, `data_path`, and `new_new_data_path` TO APPROPRIATE PATH IF ON GOOGLE COLAB

https://www.kaggle.com/datasets/gpiosenka/sports-classification

In [ ]:
DOWNLOAD = False

In [ ]:
def load_yaml(path):
    with open(path, "r") as f:
        cfg = yaml.safe_load(f)
    return cfg

def set_seed(seed: int = 42):
    random.seed(seed)                     # Python random
    np.random.seed(seed)                  # NumPy
    torch.manual_seed(seed)               # CPU
    torch.cuda.manual_seed(seed)          # GPU
    torch.cuda.manual_seed_all(seed)      # All GPUs
    torch.backends.cudnn.deterministic = True  # Deterministic convs
    torch.backends.cudnn.benchmark = False     # Disable auto-tuner for reproducibility
    print(f"Random seed set to {seed}")

def save_training_plots(
    model_name,
    loss_history,
    train_acc_history,
    val_acc_history,
    epoch_times,
    output_dir="outputs/plots"
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    epochs = np.arange(1, len(loss_history) + 1)

    # Loss plot
    plt.figure()
    plt.plot(epochs, loss_history, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_loss.png")
    plt.close()

    # Accuracy plot
    plt.figure()
    plt.plot(epochs, train_acc_history, label="Train Accuracy")
    plt.plot(epochs, val_acc_history, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Train vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_accuracy.png")
    plt.close()

    # Time per epoch plot
    plt.figure()
    plt.plot(epochs, epoch_times, label="Time per Epoch (s)")
    plt.xlabel("Epoch")
    plt.ylabel("Seconds")
    plt.title("Epoch Time")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_epoch_time.png")
    plt.close()

    print(f"\nPlots saved to: {output_dir.resolve()}")

def summarize_checkpoint_times(ckpt_path):
    ckpt = torch.load(Path("/kaggle/input/models/haruto05/resnet/pytorch/default/1") / ckpt_path, map_location="cpu")
    
    # Check if epoch_times exists
    if "epoch_times" not in ckpt:
        print("Checkpoint does not contain 'epoch_times'.")
        return None

    epoch_times = ckpt["epoch_times"]
    total_time = sum(epoch_times)
    avg_time = total_time / len(epoch_times)

    def format_hms(seconds):
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        return f"{h}h {m}m {s}s"

    print(f"Average epoch time: {format_hms(avg_time)}")
    print(f"Total training time: {format_hms(total_time)}")
    
    return avg_time, total_time

In [ ]:
RESNET_CFG = load_yaml("/kaggle/input/datasets/haruto05/resnet/resnet.yaml")
DATA_CFG = load_yaml("/kaggle/input/datasets/haruto05/resnet/data.yaml")

# Dataset

In [ ]:
def download_data(data_dir):
    data_dir = Path("/kaggle/working") / data_dir
    data_dir.mkdir(parents=True, exist_ok=True)

    download_path = kagglehub.dataset_download("gpiosenka/sports-classification")

    print("Path to dataset files:", download_path)

    return download_path

In [ ]:
if DOWNLOAD:
    download_data(DATA_CFG['root'])

In [ ]:
def build_transforms(image_size=224, train=True):
    if train:
        return T.Compose([
            T.RandomResizedCrop(image_size),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean=DATA_CFG["mean"], std=DATA_CFG["std"])
        ])
    else:
        return T.Compose([
            T.Resize(256),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize(mean=DATA_CFG["mean"], std=DATA_CFG["std"])
        ])

class SportsDataset(Dataset):
    def __init__(self, root, split="train", transform=None):
        self.root = Path(root)
        self.transform = transform

        self.df = pd.read_csv(self.root / "sports.csv")
        self.samples = self.df[self.df["data set"] == split]
        self.samples = self.samples[self.samples["filepaths"].str.endswith(".jpg")].reset_index(drop=True)

        self.classes = self.samples["labels"].unique().tolist()
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, index):
        row = self.samples.iloc[index]
        img_path = self.root / row["filepaths"]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = int(row["class id"])
        return image, label

# Model

In [ ]:
class block(nn.Module):
    def __init__(self, in_channels, out_channels, identity_downsample=None, stride=1):
        super(block, self).__init__()
        self.expansion = 4
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels*self.expansion, kernel_size=1, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(out_channels*self.expansion)
        self.relu = nn.ReLU()
        self.identity_downsample = identity_downsample

    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)
        
        x += identity
        x = self.relu(x)
        return x
    
class ResNet(nn.Module):
    def __init__(self, block, layers, image_channels, num_classes):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(image_channels, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet Layers
        self.layer1 = self._make_layer(block, layers[0], out_channels=64, stride=1)
        self.layer2 = self._make_layer(block, layers[1], out_channels=128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], out_channels=256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], out_channels=512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512*4, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc(x)

        return x

    def _make_layer(self, block, num_residual_blocks, out_channels, stride):
        identity_downsample = None
        layers = []

        if stride != 1 or self.in_channels != out_channels * 4:
            identity_downsample = nn.Sequential(nn.Conv2d(self.in_channels, out_channels*4, kernel_size=1,
                                                          stride=stride),
                                                nn.BatchNorm2d(out_channels*4))
        
        layers.append(block(self.in_channels, out_channels, identity_downsample, stride))
        self.in_channels = out_channels * 4

        for i in range(num_residual_blocks - 1):
            layers.append(block(self.in_channels, out_channels))
        
        return nn.Sequential(*layers)
    
def ResNet50(img_channels=3, num_classes=1000):
    return ResNet(block, [3, 4, 6, 3], img_channels, num_classes)
    
def ResNet101(img_channels=3, num_classes=1000):
    return ResNet(block, [3, 4, 23, 3], img_channels, num_classes)
    
def ResNet152(img_channels=3, num_classes=1000):
    return ResNet(block, [3, 8, 36, 3], img_channels, num_classes)

# Train

In [ ]:
def train(model_name):
    """
    Train a ResNet model on the sports dataset.

    Args:
        model_name (str): One of "resnet50", "resnet101", "resnet152"
    """
    # Config
    set_seed(42)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    
    # CHANGE THIS PATH TO THE DOWNLOADED DATA LOCATION IF ON GOOGLE COLAB
    data_root = "/kaggle/input/datasets/gpiosenka/sports-classification"
    
    # Datasets & loaders
    train_dataset = SportsDataset(
            root=data_root, 
            split="train", 
            transform=build_transforms(DATA_CFG["image_size"], train=True))
    val_dataset = SportsDataset(
            root=data_root, 
            split="valid", 
            transform=build_transforms(DATA_CFG["image_size"], train=False))
    
    train_loader = DataLoader(train_dataset, 
                              batch_size=DATA_CFG["batch_size"], 
                              shuffle=True, 
                              num_workers=DATA_CFG["num_workers"],
                              drop_last=True)
    val_loader = DataLoader(val_dataset, 
                              batch_size=DATA_CFG["batch_size"], 
                              shuffle=True, 
                              num_workers=DATA_CFG["num_workers"],
                              drop_last=True)
    
    # Model, loss, optimizer
    if model_name.lower() == "resnet50":
        model = ResNet50(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name.lower() == "resnet101":
        model = ResNet101(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name.lower() == "resnet152":
        model = ResNet152(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(),
        lr=float(RESNET_CFG.get("lr", 0.001)),
        momentum=float(RESNET_CFG.get("momentum", 0.9)),
        weight_decay=float(RESNET_CFG.get("weight_decay", 1e-4)),
    )
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=int(RESNET_CFG.get("step_size", 30)),
        gamma=float(RESNET_CFG.get("gamma", 0.1)),
    )
    
    # Checkpoint
    num_epochs = RESNET_CFG.get("epochs", 50)
    output_dir = Path("/kaggle/working/outputs/checkpoints")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    start_epoch = 1
    best_acc = 0.0
    
    loss_history = []
    train_acc_history = []
    val_acc_history = []
    epoch_times = []
    
    if RESNET_CFG.get("start_from", None) is not None and not isinstance(RESNET_CFG.get("start_from", None), str):
        ckpt_epoch = int(RESNET_CFG["start_from"])
        model_dir = Path(f"/kaggle/input/models/haruto05/{model_name}/pytorch/default/1")
        ckpt_path = model_dir / f"{model_name}_epoch_{ckpt_epoch}.pth"
    
        checkpoint = torch.load(ckpt_path, map_location=device)
    
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
    
        best_acc = checkpoint.get("best_acc", 0.0)
    
        loss_history = checkpoint.get("loss_history", [])
        train_acc_history = checkpoint.get("train_acc_history", [])
        val_acc_history = checkpoint.get("val_acc_history", [])
        epoch_times = checkpoint.get("epoch_times", [])
    
        start_epoch = checkpoint["epoch"] + 1
    
        print(f"Resumed from epoch {start_epoch}")
    
    # Training loop
    for epoch in range(start_epoch, num_epochs+1):
        start_time = time.time()
    
        # Training
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        for images, labels in tqdm(train_loader, desc=f"[Train] Epoch {epoch}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
    
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)
    
        epoch_loss = running_loss / len(train_loader.dataset)
        train_acc = correct_train / total_train
        loss_history.append(epoch_loss)
        train_acc_history.append(train_acc)
    
        # Validations
        model.eval()
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"[Val] Epoch {epoch}/{num_epochs}"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)
    
        val_acc = correct_val / total_val
        val_acc_history.append(val_acc)
    
        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)
    
        print(f"Epoch {epoch} | Loss: {epoch_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Time: {epoch_time:.2f}s")
    
        # Save plots
        save_training_plots(
            model_name=model_name,
            loss_history=loss_history,
            train_acc_history=train_acc_history,
            val_acc_history=val_acc_history,
            epoch_times=epoch_times,
            output_dir="outputs/plots"
        )
    
        # Save checkpoint
        ckpt = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_acc": best_acc,
        
            # histories
            "loss_history": loss_history,
            "train_acc_history": train_acc_history,
            "val_acc_history": val_acc_history,
            "epoch_times": epoch_times,
        }
    
        ckpt_path = output_dir / f"{model_name}_epoch_{epoch}.pth"
        torch.save(ckpt, ckpt_path)

        if val_acc > best_acc:
            best_acc = val_acc
            best_ckpt_path = output_dir / f"{model_name}_best.pth"
            torch.save(ckpt, best_ckpt_path)
            print(f"Saved best model to {best_ckpt_path}")
        
        scheduler.step()
    
    print("\nTraining Summary")
    print(f"Best Val Accuracy: {best_acc*100:.2f}%")
    print(f"Total time: {sum(epoch_times):.2f} seconds")
    print(f"Avg time/epoch: {np.mean(epoch_times):.2f} seconds")
    print(f"Min epoch time: {np.min(epoch_times):.2f} seconds")
    print(f"Max epoch time: {np.max(epoch_times):.2f} seconds")

model_name = "resnet50"
train(model_name)

In [ ]:
def inference(params_path, topk=(1,5)):
    model_name = params_path.split("_")[0]
    # Setup
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    
    # CHANGE THIS PATH TO THE DOWNLOADED DATA LOCATION IF ON GOOGLE COLAB
    data_root = "/kaggle/input/datasets/gpiosenka/sports-classification"

    # Create output directory for plots
    plots_dir = Path("/kaggle/working/outputs/plots")
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Create output directory for metrics
    metric_dir = Path("/kaggle/working/outputs/metrics")
    metric_dir.mkdir(parents=True, exist_ok=True)

    # Data
    test_dataset = SportsDataset(
        root=data_root, 
        split="test",  
        transform=build_transforms(DATA_CFG["image_size"], train=False)
    )
    test_loader = DataLoader(
        test_dataset, batch_size=64, shuffle=False, num_workers=1
    )

    idx_to_class = test_dataset.classes

    # Model
    if model_name == "resnet50":
        model = ResNet50(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name == "resnet101":
        model = ResNet101(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name == "resnet152":
        model = ResNet152(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        
    ckpt_path = Path("/kaggle/input/models/haruto05/resnet/pytorch/default/1") / params_path
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    # Metrics Tracking
    total = 0
    topk_correct = [0] * len(topk)
    confusion_counter = Counter()      # (true, pred)
    per_class_total = Counter()        # true
    per_class_correct = Counter()      # true & correct

    # Inference Loop
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"[Inference]"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            # Top-k accuracy
            for i, k in enumerate(topk):
                topk_preds = torch.topk(probs, k, dim=1).indices
                topk_correct[i] += (
                    topk_preds == labels.unsqueeze(1)
                ).any(dim=1).sum().item()

            # Top-1 predictions
            preds = torch.argmax(probs, dim=1)

            for t, p in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                per_class_total[t] += 1
                if t == p:
                    per_class_correct[t] += 1
                else:
                    confusion_counter[(t, p)] += 1

            total += labels.size(0)

    # Print accuracy
    print("\nAccuracy:")
    for i, k in enumerate(topk):
        acc = topk_correct[i] / total
        print(f"Top-{k}: {acc:.4f}")

    # Confusion analysis
    most_confused = confusion_counter.most_common(10)

    print("\nTop 10 most confused class pairs (true -> predicted):")
    for (t, p), count in most_confused:
        print(f"{idx_to_class[t]} -> {idx_to_class[p]} : {count}")

    if most_confused:
        # Bar plot for top 10 most confused
        labels_plot = [
            f"{idx_to_class[t]}->{idx_to_class[p]}"
            for (t, p), _ in most_confused
        ]
        counts = [c for _, c in most_confused]

        plt.figure(figsize=(10, 5))
        plt.bar(range(len(counts)), counts)
        plt.xticks(range(len(counts)), labels_plot, rotation=45)
        plt.ylabel("Count")
        plt.title("Top 10 Most Confused Class Pairs")
        plt.tight_layout()

        plot_path = plots_dir / f"{model_name}_most_confused_pairs.png"
        plt.savefig(plot_path)
        plt.close()
        print(f"\nConfusion plot saved to: {plot_path}")

        # Automatic Top-10 Confused Image Grid

        fig, axes = plt.subplots(5, 4, figsize=(18, 20))
        axes = axes.reshape(5, 4)

        for idx, ((t, p), _) in enumerate(most_confused):
            row = idx // 2
            col = (idx % 2) * 2

            true_name = idx_to_class[t]
            pred_name = idx_to_class[p]

            # Get filepaths for true and predicted classes
            t_imgs = test_dataset.samples[
                test_dataset.samples["class id"] == t
            ]["filepaths"].tolist()

            p_imgs = test_dataset.samples[
                test_dataset.samples["class id"] == p
            ]["filepaths"].tolist()

            # Sample up to 2 images per class safely
            t_sample = random.sample(t_imgs, min(2, len(t_imgs)))
            p_sample = random.sample(p_imgs, min(2, len(p_imgs)))

            # Fill 2 columns (true vs predicted)
            for i in range(2):
                if i < len(t_sample):
                    img_path = Path(data_root) / t_sample[i]
                    axes[row, col].imshow(Image.open(img_path).convert("RGB"))
                    axes[row, col].set_title(f"True: {true_name}", fontsize=9)
                    axes[row, col].axis("off")

                if i < len(p_sample):
                    img_path = Path(data_root) / p_sample[i]
                    axes[row, col + 1].imshow(Image.open(img_path).convert("RGB"))
                    axes[row, col + 1].set_title(f"Pred: {pred_name}", fontsize=9)
                    axes[row, col + 1].axis("off")

        plt.tight_layout()
        sample_img_path = plots_dir / f"{model_name}_most_confused_pairs_samples.png"
        plt.savefig(sample_img_path)
        plt.close()
        print(f"\nSample images of confused pairs saved to: {sample_img_path}")

    # Per-class accuracy CSV
    class_accuracy = []
    for cls in per_class_total:
        acc = per_class_correct[cls] / per_class_total[cls]
        class_accuracy.append(
            (cls, idx_to_class[cls], acc, per_class_correct[cls], per_class_total[cls])
        )

    # Sort high -> low accuracy
    class_accuracy.sort(key=lambda x: x[2], reverse=True)

    csv_path = metric_dir / f"{model_name}_per_class_accuracy.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_id", "class_name", "accuracy", "correct", "total"])
        for cls, name, acc, correct, total_cls in class_accuracy:
            writer.writerow([cls, name, f"{acc:.4f}", correct, total_cls])

    print(f"\nPer-class accuracy CSV saved to: {csv_path}")


if __name__ == "__main__":
    param_path = "resnet152_epoch_100.pth"
    inference(param_path)
    summarize_checkpoint_times(param_path)